# GraphRAG& Knowledge Graphs

**Module 8 · Lesson 8.7**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Knowledge Graph From Scratch - Entities + Relations
- Neo4j - Production Graph Database
- Hybrid GraphRAG - Vectors + Graph Together
- Microsoft GraphRAG - Leiden Communities, 4 Search Modes
- Cheaper Alternatives - LightRAG, LazyGraphRAG, HippoRAG 2
- LlamaIndex PropertyGraphIndex - Schema-Guided KG

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy llama-index llama-index-llms-google-genai llama-index-graph-stores-neo4j neo4j lightrag-hku transformers torch google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The Answer No Chunk Contained

> 💡 **Info**
>
> A student files a complaint about "the retrieval topic being rushed." Support needs to route it to the right instructor. The answer requires chaining four separate documents: the complaint mentions a *topic* ("retrieval"); a syllabus doc says that topic lives in a *module* ("Module 8: RAG"); a staffing doc says that module is *taught by* an instructor; a directory doc has the instructor's contact. No single chunk holds "the instructor for the topic this student complained about" - so vector search, which retrieves chunks that *resemble* the query, returns the complaint and the syllabus and stops. It has no mechanism to FOLLOW the chain complaint -> topic -> module -> instructor. **A knowledge graph does: a three-hop traversal walks those exact edges and assembles an answer that exists nowhere as text.** That is GraphRAG - and this lesson builds it from scratch, then tours the 2026 landscape that made it cheap enough to ship.

### 🎯 What you will be able to do after this lesson

- **Build** a knowledge graph from text - LLM triple extraction, BFS traversal, and a graph-grounded query - then store it in Neo4j and query with Cypher.

- **Compare** the 2026 graph-RAG landscape: Microsoft GraphRAG (Leiden communities, Local/Global/DRIFT/Basic search) vs LightRAG, LazyGraphRAG, HippoRAG 2, and PropertyGraphIndex.

- **Operate** production graph RAG: pick a database, build the hybrid vector+graph pattern, and route each query to the retriever that wins it.

- **Evaluate** graph RAG with comprehensiveness and diversity (not just NDCG/MRR), and know when GraphRAG actually loses to vanilla RAG.

> 📦 **Info**
>
> 🧭 Before you start
> You need Lessons 8.1 and 8.4 (vector + hybrid retrieval, recalled below), a Google AI Studio key in `GOOGLE_API_KEY`, and `pip install google-genai neo4j numpy` (plus `graphrag`, `lightrag-hku`, and the llama-index graph packages for the later steps). Every block uses the current **google-genai** SDK: pre-2025 tutorials import the retired `google.generativeai` and call dead models, and error on today's stack.

## 60-Second Warm-Up: What You Already Know

Three recalls - all load-bearing today. Click a card to check yourself.

> 🔗 **Analogy**
>
> **Vector RAG is searching a library by topic; GraphRAG is following a trail of footnotes.** Topic search finds books ABOUT a subject. Footnote-chasing is different: Book A names Person X, Person X wrote Paper Y, Paper Y cites Dataset Z. You TRAVERSE those citations to reach a fact no single book states. An entity is a name in the text; a relationship (a triple) is a footnote linking two of them; a knowledge graph is the whole web of footnotes. **Vectors find similar. Graphs find connected.**
> **Where the analogy breaks down:** a librarian's footnotes are written by careful humans; a GraphRAG graph is extracted by an LLM, so it inherits the LLM's errors - wrong entities, missed links, and the same thing written two ways (entity resolution). The trail is only worth following when the answer is genuinely spread across sources; for a single fact, reading the one relevant page is faster.

> 💡 **Info**
>
> ⚠️ Misconception: "GraphRAG replaces vector RAG" (and "always use GraphRAG for accuracy")
> Both wrong. GraphRAG does not replace vector RAG - on a simple fact lookup ("what is the refund window?") it is slower, costlier, and no more accurate, and on some benchmarks it UNDERPERFORMS vanilla RAG. Its power is narrow and specific: multi-hop questions and corpus-wide themes ("what are the main themes across all documents?") that similarity cannot assemble. So the production answer is not "graph everything" - it is **hybrid routing**: classify each query and send factual ones to vector RAG, relational and thematic ones to the graph. The anti-pattern to avoid is paying GraphRAG's cost on questions vector RAG already answers.

**Finds similar chunks.** Great for: factual lookup, single-doc QA, semantic match. Fails at: multi-hop reasoning, relationship queries, corpus-wide themes.

**Traverses relationships.** Great for: multi-hop questions, entity connections, cross-document themes. Needs: entity extraction, graph construction, traversal.

## Build One: A Knowledge Graph by Hand

Steps 1-3 build GraphRAG from scratch: a knowledge graph you extract and traverse yourself, Neo4j for production storage, and the hybrid pattern that combines vector breadth with graph relationship-chains.

---

## 🎯 Concept 1: Knowledge Graph From Scratch - Entities + Relations

### Knowledge Graph From Scratch - Entities + Relations

An LLM extracts (subject, relation, object) triples. BFS traverses them to answer.

**Why this is step 1:** the whole idea of GraphRAG fits in three operations you can write by hand - extract triples, traverse the graph, answer from what you gathered. Building it once, without a framework, is what makes every later tool (Microsoft GraphRAG, LightRAG, PropertyGraphIndex) legible: they are all industrial versions of this.

> 🔗 **Analogy**
>
> **You are writing the footnotes yourself.** Read each document and jot every "X relates to Y" as a note card: (Netsetos, OFFERS, GenAI Course). Pile the cards up and you have a web of connections; to answer a question, start at its entities and follow the cards a few hops out.
> **Where the analogy breaks down:** a human writes consistent cards; the LLM may write "GenAI Course" on one card and "the GenAI course" on another - two cards for one thing. That is the entity-resolution problem step 7 confronts; for now, the demo keeps names short and standardized to sidestep it.

"Which instructor teaches the topic in the student's complaint?" needs 3 hops (complaint -> topic -> module -> instructor). Can vector search answer it?

- **extract_from_text** asks the LLM for (subject, relation, object) triples and stores them in an adjacency map.

- **get_neighbors** is a BFS (breadth-first search) - start at an entity and fan outward one hop at a time, up to `hops` deep.

- **query** extracts the question's entities, traverses, and answers from the gathered triples.

**📝 `01_knowledge_graph.py`** - *google-genai*

In [ ]:
# KNOWLEDGE GRAPH FROM SCRATCH - extract triples with an LLM, traverse with BFS
# pip install google-genai
import json
from collections import defaultdict
from google import genai

client = genai.Client()   # reads GOOGLE_API_KEY from the environment

class SimpleKnowledgeGraph:
    def __init__(self):
        self.triples = []
        self.adjacency = defaultdict(list)

    def extract_from_text(self, text):
        prompt = ("Extract entity-relationship triples as a JSON array of "
                  "{subject, relation, object}. Keep entities short (1-3 words).\n\n"
                  f"Text: {text}\n\nReturn ONLY the JSON array:")
        raw = client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip()
        if raw.startswith("```"): raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
        try:
            for t in json.loads(raw):
                tri = (t["subject"], t["relation"], t["object"])
                self.triples.append(tri)
                self.adjacency[tri[0].lower()].append(tri)
                self.adjacency[tri[2].lower()].append(tri)
            return len(self.triples)
        except Exception:
            return len(self.triples)

    def get_neighbors(self, entity, hops=2):
        visited, result, queue = set(), [], [(entity.lower(), 0)]
        while queue:
            node, depth = queue.pop(0)
            if node in visited or depth > hops: continue
            visited.add(node)
            for tri in self.adjacency.get(node, []):
                result.append(tri)
                for e in (tri[0].lower(), tri[2].lower()):
                    if e not in visited: queue.append((e, depth + 1))
        return result

    def query(self, question):
        ent = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"List the key entities in this question as a JSON array of strings.\nQ: {question}").text.strip()
        if ent.startswith("```"): ent = ent.split("\n", 1)[1].rsplit("```", 1)[0]
        try: entities = json.loads(ent)
        except Exception: entities = question.lower().split()[:3]
        ctx = set()
        for e in entities:
            for s, r, o in self.get_neighbors(e, hops=2):
                ctx.add(f"{s} --[{r}]--> {o}")
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents="Answer using ONLY this knowledge graph.\nGraph:\n" + "\n".join(ctx) + f"\n\nQ: {question}\nA:").text.strip()
        return {"answer": ans, "triples_used": len(ctx)}

kg = SimpleKnowledgeGraph()
for t in [
    "Netsetos offers the GenAI course, which requires Python.",
    "The GenAI course has Module 8, which covers RAG.",
    "Module 8 (RAG) is taught by Instructor A.",
]:
    print(f"total triples: {kg.extract_from_text(t)}")
print(kg.query("Who teaches the module that covers RAG?"))

# Output:
# total triples: 2
# total triples: 4
# total triples: 5
# {'answer': 'Instructor A teaches Module 8, which covers RAG.', 'triples_used': 5}  # exact counts vary by LLM extraction

#### 💡 What just happened

⚡ What just happened? The LLM turned unstructured sentences into (subject, relation, object) triples; BFS followed the edges two hops out; the answer came from the traversed graph, not from any single sentence. **"Who teaches the module that covers RAG?" is a two-hop chain - RAG is covered by Module 8, Module 8 is taught by Instructor A - that no chunk contains and vector search cannot assemble.** The tradeoff: LLM extraction works on any domain but costs a call per document and inherits the LLM's mistakes. When to use it: whenever the answer lives in connections across documents rather than inside one.

- Pick or type a sentence and watch the extractor pull out (subject, relation, object) triples.

- Each new triple adds nodes and an edge to the growing graph - drag nodes to untangle it.

- Watch the triple and entity counters climb as you add sentences.

- See how three plain sentences become a connected web you can traverse - the whole idea of step 1, live.

Interactive: add sentences, watch the knowledge graph build itself node by node.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Neo4j - Production Graph Database

### Neo4j - Production Graph Database

Store triples in a graph database. Traverse and query with Cypher.

**A Python dict of triples works for a demo; production needs a real graph database.** Neo4j stores nodes and relationships natively and queries them with Cypher - SQL for graphs. You `MERGE` entities and edges (idempotent: no duplicates on re-insert), and you `MATCH` a path pattern to traverse. An LLM can even write the Cypher for you from a plain-English question.

> 📁 **Analogy**
>
> **Cypher is a card catalog you can query by shape.** Instead of "find cards mentioning X", you say "find a path: start at Netsetos, follow OFFERS to a course, follow HAS_MODULE to a module, return the module." The query describes the SHAPE of the trail through the cards, and Neo4j walks it.
> **Where the analogy breaks down:** a card catalog is read-only and small; Neo4j writes at scale and traverses millions of edges - but past a certain graph size, deep multi-hop traversals get expensive, a real production ceiling that step 8's database and routing choices address.

You run the same triple insert twice into Neo4j with `MERGE`. How many copies of the edge exist afterward?

- **add_triple** uses `MERGE` to insert entities and their edge without duplicating.

- **neighbours** runs a `MATCH ... [*1..hops]` variable-length path - the Cypher way to traverse.

- **text_to_cypher** has the LLM write the query from a plain-English question.

**📝 `02_neo4j_graphrag.py`** - *Neo4j*

In [ ]:
# NEO4J GRAPHRAG - production knowledge-graph storage
# pip install neo4j google-genai
# docker run -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j
from neo4j import GraphDatabase
from google import genai

client = genai.Client()

class Neo4jGraphRAG:
    def __init__(self, uri="bolt://localhost:7687", user="neo4j", password="password"):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def add_triple(self, subject, relation, obj):
        with self.driver.session() as s:
            s.run("MERGE (a:Entity {name:$subj}) "
                  "MERGE (b:Entity {name:$obj}) "
                  "MERGE (a)-[:REL {type:$rel}]->(b)",
                  subj=subject, obj=obj, rel=relation)

    def neighbours(self, entity, hops=2):
        with self.driver.session() as s:
            rows = s.run(f"MATCH p=(a:Entity {{name:$name}})-[*1..{hops}]->(b) "
                         "RETURN a.name AS src, last(relationships(p)).type AS rel, b.name AS dst",
                         name=entity)
            return [(r["src"], r["rel"], r["dst"]) for r in rows]

    def text_to_cypher(self, question):
        prompt = ("Convert the question to a Neo4j Cypher query. Schema: "
                  "(:Entity {name})-[:REL {type}]->(:Entity). Return ONLY the Cypher.\n\n"
                  f"Q: {question}\nCypher:")
        return client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip().strip("`")

# The pattern (needs a running Neo4j to execute the writes):
print("MERGE triples -> MATCH a path -> RETURN the nodes -> LLM answers\n")
for q, cypher in [
    ("What does Netsetos offer?",
     "MATCH (:Entity {name:'Netsetos'})-[:REL]->(c) RETURN c.name"),
    ("Multi-hop: what topics are in the course Netsetos offers?",
     "MATCH (:Entity {name:'Netsetos'})-[:REL]->(c)-[:REL {type:'HAS_MODULE'}]->(m) RETURN m.name"),
]:
    print(f"Q: {q}\nCypher: {cypher}\n")

# Output:
# MERGE triples -> MATCH a path -> RETURN the nodes -> LLM answers
# Q: What does Netsetos offer?  Cypher: MATCH (:Entity {name:'Netsetos'})-[:REL]->(c) RETURN c.name
# Q: Multi-hop...              Cypher: MATCH (:Entity {name:'Netsetos'})-[:REL]->(c)-[:REL {type:'HAS_MODULE'}]->(m) RETURN m.name

#### 💡 What just happened

⚡ What just happened?`MERGE` writes entities and edges idempotently (re-running never duplicates); a `MATCH p=(a)-[*1..2]->(b)` path pattern is the Cypher way to traverse N hops; and the LLM can translate a plain question into that Cypher. **The two-hop Cypher chains Netsetos to its course to its module - the same traversal as step 1's BFS, now in a database that scales.** The tradeoff: an in-memory dict is simplest for small graphs, Neo4j is persistent and query-able but is another service to run; and generated Cypher is flexible while template Cypher is safer against injection.

- Click any entity node to start a traversal from it.

- Drag the "hops" slider (1 to 3) and watch the reachable neighbourhood light up.

- The panel lists the triples on the path and how many nodes each hop reaches.

- See why a 2-hop walk from "complaint" reaches "instructor" while a 1-hop walk does not.

Interactive: pick a start node, drag the hops slider, watch BFS light the graph.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Hybrid GraphRAG - Vectors + Graph Together

### Hybrid GraphRAG - Vectors + Graph Together

Vector search for breadth, graph traversal for relationship chains. Use each where it wins.

**The best production systems do not choose vector or graph - they use both.** Vector retrieval is unbeatable for finding chunks that resemble the query (broad, fuzzy, semantic); graph traversal is unbeatable for chaining related entities (precise, connected, multi-hop). Hybrid GraphRAG runs both and hands the LLM both contexts: the similar chunks AND the relationship chain.

> 🤝 **Analogy**
>
> **Two researchers at one desk.** One skims the shelves for anything on your topic (vector - broad, fast, finds the gist); the other chases the footnotes to connect specific facts (graph - precise, follows the chain). You do not pick one; you put both reports on the desk and write the answer from the combination.
> **Where the analogy breaks down:** two researchers cost more than one - hybrid runs two retrieval systems and an extra LLM call, so it is the most accurate AND the most expensive/complex, which is exactly why step 8's router decides per query whether the second researcher is even needed.

In hybrid GraphRAG, which retriever handles "find documents similar to this complaint" vs "trace the complaint to an instructor"?

- **Vector path**: embed with `gemini-embedding-001`, cosine-rank chunks - finds SIMILAR.

- **Graph path**: reuse step 1's `SimpleKnowledgeGraph.query` - finds CONNECTED.

- Hand the LLM BOTH contexts and let it synthesize one answer.

**📝 `03_hybrid_graphrag.py`** - *Hybrid*

In [ ]:
# HYBRID GRAPHRAG - vector breadth + graph relationship-chains
# pip install google-genai numpy
import numpy as np
from google import genai
# from step 1: SimpleKnowledgeGraph

client = genai.Client()

def embed(text):
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return np.array(r.embeddings[0].values)

class HybridGraphRAG:
    def __init__(self):
        self.chunks, self.embs = [], []
        self.kg = SimpleKnowledgeGraph()   # from step 1

    def ingest(self, text):
        self.chunks.append(text); self.embs.append(embed(text))
        self.kg.extract_from_text(text)

    def query(self, question):
        qe = embed(question)
        sims = sorted(((i, float(np.dot(qe, e) / (np.linalg.norm(qe) * np.linalg.norm(e))))
                       for i, e in enumerate(self.embs)), key=lambda x: -x[1])
        vector_ctx = "\n".join(self.chunks[i] for i, _ in sims[:3])   # SIMILAR
        graph_ctx = self.kg.query(question)["answer"]                # CONNECTED
        return client.models.generate_content(model="gemini-2.5-flash",
            contents=(f"Answer using BOTH.\nVector (similar chunks):\n{vector_ctx}\n\n"
                      f"Graph (relationship chains):\n{graph_ctx}\n\nQ: {question}\nA:")).text.strip()

print("Vector finds SIMILAR chunks; graph finds CONNECTED entities; hybrid uses each where it wins.")

# Output:
# Vector finds SIMILAR chunks; graph finds CONNECTED entities; hybrid uses each where it wins.

#### 💡 What just happened

⚡ What just happened? Two retrievers, one prompt: vector cosine finds the chunks that RESEMBLE the question (broad recall), graph traversal finds the entities CONNECTED to it (the relationship chain), and the LLM writes the answer from both. **Vectors find similar, graphs find connected - hybrid is not a compromise, it is using each retriever for the half it is built for.** The tradeoff: best accuracy at the highest cost and complexity (two systems + an extra call), which is why the next arc is about when that cost is worth paying - and step 8 routes it per query.

- Pick a query - a simple factual one, or a multi-hop one.

- Toggle Vector RAG: watch it highlight similar chunks - and MISS the multi-hop answer.

- Toggle GraphRAG: watch it traverse the edges and reach the connected answer.

- The panel shows what each method retrieved and whether it actually answered.

Interactive: pick a query, toggle vector vs graph, see which one wins.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## The 2026 GraphRAG Landscape

Steps 4-10 are the industrial tooling that made GraphRAG shippable: Microsoft GraphRAG's community machinery, the cheaper alternatives, LlamaIndex's PropertyGraphIndex, the construction pipeline, production decisions, evaluation, and the multilingual stack. Each carries runnable code.

---

## 🎯 Concept 4: Microsoft GraphRAG - Leiden Communities, 4 Search Modes

### Microsoft GraphRAG - Leiden Communities, 4 Search Modes

Cluster the graph into communities, summarize each, then search local, global, or DRIFT.

**Microsoft GraphRAG's killer feature is community summaries.** After extracting the graph, it runs Leiden community detection to cluster related entities, then has an LLM write a natural-language summary of each community. That unlocks a question no chunk and no single traversal can answer: "what are the main themes across ALL documents?" - Global Search map-reduces (summarize each community, then combine those summaries) over the pre-computed community summaries. The cost: entity extraction dominates indexing, which is why the tool is CLI-first and offers a cheaper `--method fast`.

> 📚 **Analogy**
>
> **Communities are the chapters of the footnote web, each with an abstract.** A pile of note cards is hard to summarize; group them into topical clusters (a "chapter") and write an abstract per cluster, and now "what are the big themes?" is answered by reading the abstracts, not every card. Local search reads one card's neighbourhood; Global search reads all the abstracts; DRIFT reads an abstract to decide which cards to chase.
> **Where the analogy breaks down:** a human clusters by judgement; Leiden clusters by graph structure (densely-connected groups), so a community is "entities that link to each other a lot", which may not match a human's sense of a theme - and building all those abstracts up front is the expensive step LazyGraphRAG defers.

"What are the main themes across all 10,000 documents?" Which GraphRAG search mode is built for it?

- `graphrag init` then `graphrag index` runs the 6-phase pipeline (the code prints them).

- `--method fast` uses NLP instead of an LLM for extraction - far cheaper indexing.

- `graphrag query --method [local|global|drift|basic]` picks the search strategy.

**📝 `04_microsoft_graphrag.py graphrag`** - *CLI*

In [ ]:
# MICROSOFT GRAPHRAG - the CLI pipeline (Leiden communities + 4 search modes)
# pip install graphrag
#
# 1) Initialize a workspace (writes settings.yaml + .env)
#    graphrag init --root ./ragtest
#
# 2) Point settings.yaml at your model, then index. --method fast = NLP extraction (cheaper)
#    graphrag index --root ./ragtest --method fast
#
# 3) Query with one of four search methods (default is global)
#    graphrag query --root ./ragtest --method local  "Who teaches the RAG module?"
#    graphrag query --root ./ragtest --method global "What are the main themes?"
#    graphrag query --root ./ragtest --method drift  "How does RAG relate to fine-tuning?"

phases = [
    "1. Chunk text into ~1,200-token units",
    "2. LLM extracts entities + relationships + claims  (the bulk of the cost)",
    "3. Leiden community detection clusters the graph (hierarchical)",
    "4. LLM writes a natural-language summary for each community",
    "5. Link communities back to their source text units",
    "6. Embed entities + community reports into a vector store",
]
for p in phases:
    print(p)
print("\nGlobal Search then map-reduces over the community summaries at query time.")

# Output:
# 1. Chunk text into ~1,200-token units
# 2. LLM extracts entities + relationships + claims  (the bulk of the cost)
# ... (6 phases) ...
# Global Search then map-reduces over the community summaries at query time.

#### 💡 What just happened

⚡ What just happened? Microsoft GraphRAG is a CLI framework (`init` / `index` / `query`). Indexing runs the 6-phase pipeline; the payoff is **Leiden communities + LLM community summaries**, which Global Search map-reduces to answer corpus-wide thematic questions no chunk holds. **Local search is entity-centric, Global is theme-centric, DRIFT blends them (a global primer that seeds local traversal), and Basic is plain vector top-k.** The tradeoff: entity extraction is the bulk of indexing cost, so use `--method fast` (NLP extraction) when the budget is tight - and the whole point of the next step is that others made this dramatically cheaper.

- Drag the resolution slider and watch Leiden-style community detection recolour the graph into clusters.

- Click a community to read its LLM-written summary.

- Toggle Local vs Global search and watch which nodes each one touches.

- See why Global (reading all the summaries) answers themes that Local (one neighbourhood) cannot.

Interactive: cluster the graph, read a community summary, compare local vs global reach.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Cheaper Alternatives - LightRAG, LazyGraphRAG, HippoRAG 2

### Cheaper Alternatives - LightRAG, LazyGraphRAG, HippoRAG 2

Skip the expensive community pre-compute, or defer graph work to query time.

**Full GraphRAG front-loads cost into indexing; the 2026 alternatives move that cost around.** LightRAG skips community detection for a dual-level (entity + topic) retrieval and updates incrementally. LazyGraphRAG (Microsoft) does the graph work LAZILY at query time, so indexing costs the same as plain vector RAG. HippoRAG 2 uses Personalized PageRank for cheap multi-hop. The decision is about your data: stable and queried often, or changing and cost-sensitive.

> 🏦 **Analogy**
>
> **Prepay or pay-as-you-go.** Microsoft GraphRAG is a prepaid annual pass: expensive up front (build every community summary), then cheap per visit - great if you go often and the museum does not change. LazyGraphRAG is a day pass bought at the door: nothing up front, you pay a little each visit - great if you rarely go or the exhibits change weekly.
> **Where the analogy breaks down:** a museum pass is a fixed menu; these frameworks differ in what they can ANSWER, not just price - Microsoft's summaries capture corpus-wide themes better, while LightRAG handles incremental updates natively. Cost is one axis; query type and data churn are the others.

Your document set changes daily and you are cost-sensitive. Full Microsoft GraphRAG or a lazy/incremental alternative?

- Wire your own LLM and embedding functions (here, Gemini) into `LightRAG(...)`.

- `ainsert` builds the dual-level graph; it supports incremental inserts.

- `aquery(..., QueryParam(mode=...))` picks naive / local / global / hybrid retrieval.

**📝 `05_lightrag.py`** - *LightRAG*

In [ ]:
# LIGHTRAG - dual-level retrieval, cheap indexing, incremental updates
# pip install lightrag-hku google-genai numpy
import asyncio
import numpy as np
from google import genai
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status

client = genai.Client()

# Adapt Gemini to LightRAG's function interfaces
async def gemini_complete(prompt, system_prompt=None, history_messages=[], **kwargs):
    return client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text

def gemini_embed(texts):
    r = client.models.embed_content(model="gemini-embedding-001", contents=texts)
    return np.array([e.values for e in r.embeddings])

async def main():
    rag = LightRAG(
        working_dir="./lightrag_store",
        llm_model_func=gemini_complete,
        embedding_func=EmbeddingFunc(embedding_dim=3072, max_token_size=8192, func=gemini_embed),
    )
    await rag.initialize_storages()
    await initialize_pipeline_status()

    await rag.ainsert("Netsetos offers a GenAI course; Module 8 covers RAG and GraphRAG.")
    for mode in ["naive", "local", "global", "hybrid"]:
        ans = await rag.aquery("What does Module 8 cover?", param=QueryParam(mode=mode))
        print(f"[{mode:7s}] {ans[:56]}")

asyncio.run(main())

# Output:
# [naive  ] top-k chunks, no graph
# [local  ] entity-level neighbourhood
# [global ] topic-level themes
# [hybrid ] both levels merged  (entity + topic)

#### 💡 What just happened

⚡ What just happened? LightRAG exposes the same insert/query shape as any RAG, but its graph does DUAL-LEVEL retrieval (entity-level for specifics, topic-level for themes) and supports incremental `ainsert` without a full reindex. **LazyGraphRAG (Microsoft) goes further - it indexes at vector-RAG cost and does the graph work lazily at query time, reportedly matching Global-Search quality at far lower query cost.** When to use which: stable data queried often - full GraphRAG amortizes; changing data or a tight budget - LazyGraphRAG/LightRAG; heavy multi-hop - HippoRAG 2's PageRank. Cost is the tradeoff axis, but query type and data churn decide alongside it (see [Microsoft's LazyGraphRAG post](https://www.microsoft.com/en-us/research/blog/lazygraphrag-setting-a-new-standard-for-quality-and-cost/) and the [LightRAG repository](https://github.com/HKUDS/LightRAG)).

- Drag the corpus-size slider and the monthly-query-volume slider.

- Watch the indexing and query cost bars for Vector / GraphRAG / LightRAG / LazyGraphRAG update live.

- See GraphRAG's tall indexing bar shrink to near-zero for LazyGraphRAG.

- The widget recommends a framework for your corpus size and query load.

Interactive: set corpus size + query volume, compare per-framework cost, get a recommendation.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: LlamaIndex PropertyGraphIndex - Schema-Guided KG

### LlamaIndex PropertyGraphIndex - Schema-Guided KG

The 8.6 forward hook: labeled property graphs with schema-constrained extraction.

**PropertyGraphIndex is LlamaIndex's graph path - and it replaced the deprecated KnowledgeGraphIndex.** The old index stored bare triples; PropertyGraphIndex stores LABELED nodes (PERSON, COURSE) with arbitrary properties (dates, confidence) and rich edges. Its `SchemaLLMPathExtractor` constrains extraction to an ontology you define - with `strict=True`, the LLM can only produce entities and relations you allow, so the graph stays clean.

> 🏷️ **Analogy**
>
> **A labeled index card beats a bare note.** Step 1's cards said "(GenAI Course, COVERS, RAG)". A property-graph card is labeled and annotated: "COURSE: GenAI Course [price 14999, hours 146] --COVERS--> TOPIC: RAG [difficulty: intermediate]". The label and properties let retrieval filter ("courses under 15000 that cover RAG"), which bare triples cannot.
> **Where the analogy breaks down:** richer cards take more effort to fill - schema-constrained extraction is more prompt engineering and can DROP facts that do not fit the ontology, so `strict=True` trades recall for cleanliness. Start with open extraction (SimpleLLMPathExtractor), then formalize.

You set `SchemaLLMPathExtractor(strict=True)` with a fixed list of entity/relation types. What happens to a fact whose relation is not in your list?

- Define the ontology: allowed entity types, relation types, and which relations each entity can have.

- `SchemaLLMPathExtractor(strict=True)` constrains the LLM to that ontology.

- `PropertyGraphIndex.from_documents` with a `Neo4jPropertyGraphStore`; then `as_retriever`.

**📝 `06_property_graph.py`** - *PropertyGraphIndex*

In [ ]:
# LLAMAINDEX PROPERTYGRAPHINDEX - schema-guided KG (replaces KnowledgeGraphIndex)
# pip install llama-index llama-index-graph-stores-neo4j llama-index-llms-google-genai
from typing import Literal
from llama_index.core import PropertyGraphIndex, Document, Settings
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.llms.google_genai import GoogleGenAI

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")

# Constrain extraction to an ontology - strict=True keeps the graph clean
entities = Literal["COMPANY", "COURSE", "TOPIC", "PERSON"]
relations = Literal["OFFERS", "COVERS", "REQUIRES", "TAUGHT_BY"]
schema = {"COMPANY": ["OFFERS"], "COURSE": ["COVERS", "REQUIRES"], "TOPIC": ["TAUGHT_BY"]}

extractor = SchemaLLMPathExtractor(
    llm=Settings.llm, possible_entities=entities,
    possible_relations=relations, kg_validation_schema=schema, strict=True,
)

docs = [Document(text="Netsetos offers the GenAI course, which covers RAG and requires Python.")]
index = PropertyGraphIndex.from_documents(
    docs, kg_extractors=[extractor],
    property_graph_store=Neo4jPropertyGraphStore(username="neo4j", password="password", url="bolt://localhost:7687"),
)
retriever = index.as_retriever(include_text=True, similarity_top_k=2)
print(retriever.retrieve("What does the GenAI course cover?"))

# Output: a list of NodeWithScore over the matching labeled subgraph, e.g.
# (Netsetos:COMPANY)-[OFFERS]->(GenAI course:COURSE)-[COVERS]->(RAG:TOPIC)

#### 💡 What just happened

⚡ What just happened? PropertyGraphIndex builds a LABELED property graph - nodes carry types (COMPANY, COURSE) and properties, not just names - and `SchemaLLMPathExtractor(strict=True)` forces the LLM to conform to your ontology. **Use PropertyGraphIndex when you want fine-grained control over extraction and retrieval; use Microsoft GraphRAG when you want out-of-the-box community summaries for global themes.** The tradeoff: strict schema keeps the graph clean but drops out-of-schema facts (less recall), so start with open extraction and formalize the schema for production. This is the 8.6 forward hook, now built.

---

## 🎯 Concept 7: Graph Construction - Extraction, Schema, Entity Resolution

### Graph Construction - Extraction, Schema, Entity Resolution

The graph is only as good as its construction. Entity resolution is the hidden bottleneck.

**Every retrieval quality problem downstream traces back to construction.** Three decisions define it: how you EXTRACT (a full LLM is accurate but costly; a small zero-shot NER like GLiNER is fast and cheap but narrower), how you design the SCHEMA (start emergent, formalize for production), and how you RESOLVE entities (merge "GenAI Course" and "the GenAI course" into one node). Entity resolution is the underrated one - unresolved duplicates silently split the graph so traversals miss links.

> ✍️ **Analogy**
>
> **Standardize the names before you file.** A library where "J.K. Rowling", "JK Rowling", and "Rowling, J.K." are three different author cards will scatter one author's books across three drawers. Entity resolution is the librarian who merges those into one canonical name so every book is findable together.
> **Where the analogy breaks down:** a librarian resolves by hand; at scale you resolve with embeddings (similar names are candidate duplicates) plus an LLM to verify ambiguous pairs - and over-merging is its own error (two real people named "A. Kumar" are NOT the same node), so resolution is a precision/recall balance, not a clean fix.

Your graph has "GenAI Course" and "the GenAI course" as two separate nodes. What breaks?

- Embed each entity mention with `gemini-embedding-001`.

- High cosine similarity flags CANDIDATE duplicates.

- In production, an LLM verifies the ambiguous pairs before merging.

**📝 `07_construction.py`** - *google-genai*

In [ ]:
# GRAPH CONSTRUCTION - entity resolution (the underrated bottleneck)
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()

def embed(t):
    return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Merge duplicate mentions before they fragment the graph
mentions = ["GenAI Course", "the GenAI course", "GenAI programme", "Data Science Course"]
vecs = [embed(m) for m in mentions]
for i in range(len(mentions)):
    for j in range(i + 1, len(mentions)):
        if cos(vecs[i], vecs[j]) > 0.88:          # candidate duplicate -> LLM verifies in prod
            print(f"likely same entity: '{mentions[i]}' == '{mentions[j]}'")

# Output:
# likely same entity: 'GenAI Course' == 'the GenAI course'
# likely same entity: 'GenAI Course' == 'GenAI programme'

#### 💡 What just happened

⚡ What just happened? Embedding similarity flags "GenAI Course", "the GenAI course", and "GenAI programme" as the same entity before they fragment the graph - an LLM verifies the ambiguous pairs, then you merge them into one canonical node. **Extraction gets the attention, but entity resolution is what silently determines retrieval quality: duplicate names split edges so traversals miss links.** When to use a small NER over an LLM: high-volume, narrow-domain extraction where cost matters (GLiNER-style zero-shot or spaCy); use the LLM for open-domain relation extraction. LightRAG dedups automatically, which is part of why it updates cheaply.

---

## 🎯 Concept 8: Production - Databases, Decision Framework, Hybrid Routing

### Production - Databases, Decision Framework, Hybrid Routing

The biggest decision is not which GraphRAG - it is whether to graph this query at all.

**In production, GraphRAG is not a default - it is a routed choice.** On simple factual lookups, vector RAG is faster, cheaper, and often more accurate; on multi-hop and thematic queries, the graph wins. So the highest-leverage production pattern is a router: classify each incoming query and send it to the retriever that wins it. Around that sit the database choices (Neo4j for the ecosystem, FalkorDB for low latency, NebulaGraph for super-scale, NetworkX for prototypes).

> 🏥 **Analogy**
>
> **The router is a triage nurse.** Not every patient needs the specialist; the nurse reads the symptoms and sends simple cases to the fast general clinic (vector RAG) and complex, multi-system cases to the specialist (graph RAG). Triage is quick and cheap, and it stops you from sending every headache to neurology.
> **Where the analogy breaks down:** a nurse is rarely wrong; an LLM classifier sometimes mis-routes, so you tune the threshold and accept that a misrouted query just costs a bit more or answers slightly worse - the router adds latency and a small error rate, which is the price of not paying GraphRAG's cost on every query.

"What is the refund policy?" arrives at your hybrid system. Where should the router send it, and why?

- An LLM classifies the query as *factual* or *relational/thematic*.

- Factual goes to vector RAG (fast, cheap); relational goes to graph RAG.

- The router is cheap relative to running the wrong (or both) retrievers.

**📝 `08_router.py`** - *google-genai*

In [ ]:
# HYBRID ROUTER - send each query to the retriever that wins it
# pip install google-genai
from google import genai

client = genai.Client()

def route(question):
    prompt = ("Classify the question as 'factual' (a single fact / lookup) or "
              "'relational' (multi-hop, cross-document, or 'main themes'). One word.\n\n"
              f"Q: {question}\nClass:")
    kind = client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip().lower()
    return "vector_rag" if "factual" in kind else "graph_rag"

for q in ["What is the refund policy?",
          "What are the main themes across all support tickets?",
          "Who teaches the module that covers the topic in this complaint?"]:
    print(f"{route(q):10s} <- {q}")

# Output:
# vector_rag <- What is the refund policy?
# graph_rag  <- What are the main themes across all support tickets?
# graph_rag  <- Who teaches the module that covers the topic in this complaint?

#### 💡 What just happened

⚡ What just happened? A one-call classifier routes each query: the refund lookup goes to vector RAG (fast, cheap, accurate for single facts), while the thematic and multi-hop questions go to graph RAG. **This is the single most important production decision in the lesson - not which GraphRAG framework, but whether to graph a given query at all - because GraphRAG can underperform vanilla RAG on simple facts.** The tradeoff: the router adds a little latency and a small mis-route rate, in exchange for not paying GraphRAG's cost on every query. On database choice: Neo4j is the default (biggest ecosystem, managed AuraDB), FalkorDB for low latency, NebulaGraph for super-scale, NetworkX for prototypes under ~100K nodes.

- Type or pick a query - factual, multi-hop, or thematic.

- Watch the classifier route it to Vector RAG or GraphRAG.

- See the signal words it keyed on and a confidence gauge move.

- Try edge cases and watch the route (and the reason) change.

Interactive: send queries through the router, see where each one goes and why.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 9: GraphRAG Evaluation - Comprehensiveness, Diversity, LLM-as-Judge

### GraphRAG Evaluation - Comprehensiveness, Diversity, LLM-as-Judge

The vector-RAG metrics miss what GraphRAG is for. Measure differently.

**The order-aware metrics from 8.4 (NDCG, MRR) score whether the right chunk ranked high - but GraphRAG's value is comprehensiveness and diversity, which those metrics cannot see.** Microsoft's framework judges four dimensions with an LLM: comprehensiveness (coverage), diversity (variety of perspectives), empowerment (does it help the user understand), and directness (conciseness) - via pairwise LLM-as-judge comparison against a baseline.

> ⚖️ **Analogy**
>
> **A debate judge, not a stopwatch.** NDCG/MRR are stopwatches - they measure whether the right runner finished first. But "what are the main themes?" has no single right chunk to rank; you need a judge who reads two full answers and rules which is more comprehensive and covers more angles. That is LLM-as-judge pairwise comparison.
> **Where the analogy breaks down:** a human judge is consistent; an LLM judge has variance and biases (it can favour longer or first-shown answers), so you randomize order, use a strong judge model, and average several comparisons - the metric itself needs its own guardrails.

Why do NDCG and MRR (from 8.4) fail to evaluate a GraphRAG answer to "what are the main themes?"

- Give the judge the question and two candidate answers.

- Ask which is better on comprehensiveness / diversity / empowerment / directness.

- In production, randomize A/B order and average several runs to cut bias.

**📝 `09_eval.py`** - *google-genai*

In [ ]:
# LLM-AS-JUDGE - GraphRAG needs comprehensiveness/diversity, not NDCG/MRR
# pip install google-genai
from google import genai

client = genai.Client()

def judge(question, answer_a, answer_b):
    prompt = ("Compare two answers on comprehensiveness, diversity, empowerment, and "
              "directness. Say which is better overall (A or B) and why in one line.\n\n"
              f"Q: {question}\n\nA: {answer_a}\n\nB: {answer_b}\n\nVerdict:")
    return client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip()

print(judge(
    "What are the main themes across the course?",
    "The course covers RAG.",                                              # narrow (vector-ish)
    "Themes: retrieval systems, model architecture, prompting, and production ops.",  # broad (graph-ish)
))

# Output:
# B is more comprehensive: it surfaces cross-document themes, not a single chunk's fact.

#### 💡 What just happened

⚡ What just happened? An LLM judge read two answers and ruled the broad one more comprehensive - the kind of judgement NDCG and MRR (which score a known chunk's rank) cannot make. **GraphRAG-Bench and Microsoft's four-dimension framework exist because graph answers must be judged on coverage and diversity, not just retrieval rank - and the critical finding is that GraphRAG can UNDERPERFORM vanilla RAG on simple fact retrieval, so you evaluate both.** The tradeoff: LLM-as-judge captures what matters but has variance and position bias, so randomize order and average - the full harness and CI gates are Lesson 8.11's.

---

## 🎯 Concept 10: Multilingual + India GraphRAG - Legal KGs, Hindi NER

### Multilingual + India GraphRAG - Legal KGs, Hindi NER

Domain and Indic NER beat generic LLMs where the language and jargon are specialized.

**Graph construction in Hindi and specialized domains needs specialist extractors, not a generic LLM.** Hindi has no capitalization to signal names, free word order, and rich morphology, so generic NER underperforms; MuRIL and IndicNER (AI4Bharat) are trained for it. For Indian legal graphs, OpenNyAI extracts legal entity types (Court, Judge, Statute, Provision), InLegalBERT provides legal-domain embeddings, and NyayGraph is a ready IPC knowledge graph on Neo4j.

> 📑 **Analogy**
>
> **You call a specialist notary for a legal deed, not a general clerk.** A general clerk (a generic LLM) reads everyday text well but mislabels "Section 420 IPC" or a Devanagari name; the specialist notary (OpenNyAI, MuRIL) is trained on exactly that jargon and script and gets it right. Use the specialist where the language or domain is specialized.
> **Where the analogy breaks down:** a specialist is narrow - a legal NER knows nothing about, say, cooking recipes - so the production pattern is often HYBRID: a domain NER for the entities it is trained on, plus an LLM for open relations, rather than one model for everything.

You are extracting entities from Hindi court judgments. Generic GPT-style LLM, or a domain/Indic NER?

- Load AI4Bharat's `IndicNER` via a Hugging Face pipeline.

- Run it on Devanagari text; it tags ORG/LOC/PER where a generic model has no cues.

- For legal graphs, swap in OpenNyAI's legal NER and InLegalBERT embeddings.

**📝 `10_india.py`** - *Multilingual*

In [ ]:
# MULTILINGUAL + INDIA GRAPHRAG - domain/Indic NER beats a generic LLM here
# pip install transformers torch
from transformers import pipeline

# IndicNER handles Hindi entities where a generic model has no capitalization cues
ner = pipeline("token-classification", model="ai4bharat/IndicNER", aggregation_strategy="simple")
text = "नेटसेटोस हैदराबाद में जेनएआई कोर्स प्रदान करता है।"   # Netsetos offers a GenAI course in Hyderabad
for ent in ner(text):
    print(ent["word"], "->", ent["entity_group"])

# For Indian legal graphs: OpenNyAI (Court, Judge, Statute, Provision) + InLegalBERT
# embeddings + NyayGraph (IPC knowledge graph on Neo4j).

# Output:
# नेटसेटोस -> ORG
# हैदराबाद -> LOC

#### 💡 What just happened

⚡ What just happened? IndicNER tagged the Hindi ORG and LOC that a generic English NER would miss - Devanagari has no capitalization to signal a name, so a model trained on Indic text wins. **For Indian legal graph RAG the stack is specialized: OpenNyAI for legal entity types, InLegalBERT for embeddings, NyayGraph for a ready IPC graph, and MuRIL/IndicNER for Hindi.** The tradeoff: a specialist NER is narrow, so production often HYBRIDIZES - a domain model for the entities it knows plus an LLM for open relations - rather than forcing one model to do everything. Cross-lingual graphs then either translate-first (simpler) or keep parallel KGs (preserves nuance).

## Putting It Together

You built a knowledge graph by hand, stored it in Neo4j, and hybridized it with vectors - then toured the tooling that made graph RAG shippable: Microsoft GraphRAG's communities, the cheaper LightRAG/LazyGraphRAG variants, PropertyGraphIndex, the construction pipeline, the production router, and evaluation. The through-line: *vectors find similar, graphs find connected* - and the discipline is knowing which one a given query needs.

> 📦 **Info**
>
> 🔑 Key takeaways
> - **Similar vs connected.** Vector RAG retrieves chunks that resemble the query; GraphRAG traverses relationships to chain facts across documents - the only way to answer multi-hop and corpus-wide questions.
> - **Build is three operations.** Extract (subject, relation, object) triples, traverse with BFS (or Cypher in Neo4j), answer from what you gathered.
> - **Communities enable global.** Leiden clusters + LLM community summaries are what let Microsoft GraphRAG's Global Search answer "what are the main themes?".
> - **Prepay or defer.** Full GraphRAG front-loads indexing cost (amortizes on stable data); LazyGraphRAG/LightRAG defer or update incrementally (win on churn/budget).
> - **Construction is the floor.** The graph inherits the LLM's extraction errors, and entity resolution (merging duplicate names) silently decides retrieval quality.
> - **Route, do not graph everything.** GraphRAG can lose to vector RAG on simple facts, so classify each query and send it to the retriever that wins it. Evaluate with comprehensiveness/diversity, not just NDCG/MRR.

> ✅ **Info**
>
> 🗺️ Where this goes next
> - In Lesson 8.10 (**Agentic RAG**) an agent decides at runtime WHEN to traverse the graph and HOW deep - step 8's router becomes an agentic decision.
> - In Lesson 8.11 (**RAG Evaluation**) step 9's LLM-as-judge becomes a golden-set + CI harness that gates every change.
> - In Module 14 (**LLMOps**) the expensive indexing pipeline becomes a monitored, scheduled production job.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each builds or operates one layer of the graph-RAG stack.

---

## 🎓 Summary

You've completed the practical part of **GraphRAG& Knowledge Graphs**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_7.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.7.html` - regenerate this notebook whenever the source HTML is updated.*
